# Import Libraries

In [ ]:
import torch
from torch import nn, optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import time
from torchvision import datasets, transforms, models

import timm
import pandas as pd
import os
from glob import glob
import numpy as np
from torch.utils.data import DataLoader
from torch.utils.data.sampler import SubsetRandomSampler
import torchvision.transforms as transforms
from torchvision import datasets
import pandas as pd
from PIL import Image
from skimage import io, img_as_ubyte
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms

np.random.seed(0)

# Model

In [2]:
np.random.seed(0)

class ResNetSimCLR(nn.Module):

    def __init__(self, base_model, out_dim):
        super(ResNetSimCLR, self).__init__()
        self.resnet_dict = {"resnet18": models.resnet18(pretrained=False, norm_layer=nn.InstanceNorm2d),
                            "resnet50": models.resnet50(pretrained=False)}

        resnet = self._get_basemodel(base_model)
        num_ftrs = resnet.fc.in_features

        self.features = nn.Sequential(*list(resnet.children())[:-1])

        # projection MLP
        self.l1 = nn.Linear(num_ftrs, num_ftrs)
        self.l2 = nn.Linear(num_ftrs, out_dim)

    def _get_basemodel(self, model_name):
        try:
            model = self.resnet_dict[model_name]
            print("Feature extractor:", model_name)
            return model
        except:
            raise ("Invalid model name. Check the config file and pass one of: resnet18 or resnet50")

    def forward(self, x):
        h = self.features(x)
        h = h.squeeze()

        x = self.l1(h)
        x = F.relu(x)
        x = self.l2(x)
        return h, x
    
class SimCLRClassifier(nn.Module):
    def __init__(self, encoder, num_classes):
        super(SimCLRClassifier, self).__init__()
        self.encoder = encoder
        self.classifier = nn.Sequential(
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        features, _ = self.encoder(x)
        out = self.classifier(features)
        return out
    
def load_pretrained_simclr_model(model_path):
    model = ResNetSimCLR(base_model= "resnet50",out_dim=2 )# .to(self.device)
    state_dict = torch.load(model_path)
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    return model


In [4]:
model_path  = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/Antibodies_detection/codes/tmi2022/feature_extractor/runs/May30_10-28-40_kif-gh200-04.gladstone.internal/checkpoints/model.pth"

In [3]:
model_path  = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/Antibodies_detection/codes/tmi2022/feature_extractor/runs/May23_19-38-08_kif-gh200-02.gladstone.internal/checkpoints/model.pth"

In [4]:
input_shape = (224,224,3)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
simclr_model = load_pretrained_simclr_model(model_path)

In [6]:
model = SimCLRClassifier(simclr_model, num_classes=2)

In [8]:
#model2_path = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/crop_classifier/saved_models/model.pth"
#model2_path = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/crop_classifier/saved_models/using_resnet1_model.pth"
model2_path = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/crop_classifier/saved_models/using_resnet1_model_train_val_new.pth"

In [ ]:
state_dict2 = torch.load(model2_path)
model.load_state_dict(state_dict2)
model.to(device)

In [ ]:
model.encoder.features[-2]


## Run GradCAM

In [11]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model.eval()
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None
        self.hook_handles = []

        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output.detach()

        def backward_hook(module, grad_in, grad_out):
            self.gradients = grad_out[0].detach()

        self.hook_handles.append(self.target_layer.register_forward_hook(forward_hook))
        self.hook_handles.append(self.target_layer.register_backward_hook(backward_hook))

    def __call__(self, input_tensor, class_idx=None):
        output = self.model(input_tensor)
        
        if class_idx is None:
            class_idx = output.argmax(dim=0).item() #dim=1 for batch

        self.model.zero_grad()
        
        class_score =  output[class_idx]            #output[:, class_idx]
        
        class_score.backward(retain_graph=True)

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)

        cam = F.interpolate(cam, size=input_tensor.shape[2:], mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)  # Normalize
        return cam

    def clear_hooks(self):
        for handle in self.hook_handles:
            handle.remove()

In [12]:
#model = SimCLRClassifier(encoder, 2)
target_layer = model.encoder.features[-2]  # Use the last conv block of ResNet

gradcam = GradCAM(model, target_layer)

In [ ]:
# running all predictions, one time thing
df = pd.read_csv("/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/crop_classifier/all_patches_oxford_updated.csv")
transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor()
])
"""
for idx, (img_path,label) in enumerate(zip(df["path"].values[177+9:],df["label"].values[177+9:])):
    img = Image.open(img_path).convert("RGB")
    img_resized = img.resize((512, 512))
    input_tensor = transform(img).unsqueeze(0).to(device)
    if model(input_tensor).argmax().cpu().numpy() != label:
        print(idx)
        print(img_path, label)
        
        
preds = dict()
for img_path in df["path"].values:
    img = Image.open(img_path).convert("RGB")
    img_resized = img.resize((512, 512))
    input_tensor = transform(img).unsqueeze(0).to(device)
    preds[img_path] = model(input_tensor).argmax().cpu().numpy()
    
df_pred = pd.DataFrame({"path":preds.keys(), "preds":preds.values()})

df1 = pd.merge(df,df_pred, on="path",how="left")
df1.to_csv("/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/crop_classifier/preds.csv")
"""

In [ ]:
# running all predictions, one time thing
df = pd.read_csv("/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/crop_classifier/val.csv")
transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor()
])
preds = dict()
for img_path in df["path"].values:
    img = Image.open(img_path).convert("RGB")
    img_resized = img.resize((512, 512))
    input_tensor = transform(img).unsqueeze(0).to(device)
    preds[img_path] = model(input_tensor).argmax().cpu().numpy()
    
df_pred = pd.DataFrame({"path":preds.keys(), "preds":preds.values()})

df1 = pd.merge(df,df_pred, on="path",how="left")
df1.to_csv("/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/crop_classifier/preds_val.csv")

In [15]:
def load_image(img_path):
    img = Image.open(img_path).convert("RGB")
    img_resized = img.resize((512, 512))
    img_np = np.array(img_resized) / 255.0  # [H, W, 3], float32 in range [0, 1]
    input_tensor = transform(img).unsqueeze(0).to(device)
    return img_resized, img_np, input_tensor
    
def load_image_run_gradcam(img_path, model):
    # 1. Load and preprocess the image
    img_resized, img_np, input_tensor = load_image(img_path)
    print(model(input_tensor))
    cam = gradcam(input_tensor)  # cam should be [H, W], values in [0, 1]
    # 3. Convert CAM to heatmap
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)  # [H, W, 3], BGR
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB) / 255.0  # Convert to RGB + normalize
    alpha = 0.7

    # 4. Overlay heatmap onto image
    overlay = alpha * img_np + (1-alpha) * heatmap
    overlay = np.clip(overlay, 0, 1)
    return img_resized,overlay
    
def load_image_run_saliency_map(img_path, model):
    # 1. Load and preprocess the image
    img_resized, img_np, input_tensor = load_image(img_path)
    input_tensor.requires_grad_()
    output = model(input_tensor)
    class_idx = output.argmax()
    # Backward pass for that class
    model.zero_grad()
    output[class_idx].backward()

    saliency = input_tensor.grad.data.abs().max(dim=1)[0]
    saliency = input_tensor.grad.data.abs()  # absolute value of gradients
    saliency, _ = torch.max(saliency, dim=1)  # Take max along channel dimension
    saliency = saliency.squeeze().cpu().numpy()  # [H, W]
    saliency = (saliency - saliency.min()) / (saliency.max() - saliency.min())
    saliency = np.uint8(saliency * 255)
    saliency_colored = cv2.applyColorMap(saliency, cv2.COLORMAP_JET)
    saliency_colored = cv2.cvtColor(saliency_colored, cv2.COLOR_BGR2RGB)

    alpha = 0.7

    # 4. Overlay heatmap onto image
    overlayed = alpha * img_np + (1-alpha) * saliency_colored
    overlayed = np.clip(overlay, 0, 1)
    return img_resized,overlayed

def save_gradcam(img_path,save_path, cmap='jet'):
    img_resized,gradcam = load_image_run_gradcam(img_path, model)
    _, ablationcam  = load_img_apply_ablationcam(img_path, model)
    plt.figure(figsize=(18, 6))
    title1 = "Image"
    # First image
    plt.subplot(1, 3, 1)
    plt.imshow(img_resized, cmap='jet')
    plt.title(title1)
    plt.axis('off')
    title2 = "Grad-cam overlay"
    # Second image
    plt.subplot(1, 3, 2)
    plt.imshow(gradcam, cmap='jet')
    plt.title(title2)
    plt.axis('off')
    plt.tight_layout()
    # Third image
    title3 = "Ablation-cam overlay"
    plt.subplot(1, 3, 3)
    plt.imshow(ablationcam, cmap='jet')
    plt.title(title3)
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')  # Save before plt.show()
    


In [ ]:
df1= pd.read_csv("/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/crop_classifier/preds_val.csv")
save_path_dir =  "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/crop_classifier/DLB_correct_preds_val"
for i in range(25):
    ind = np.random.choice(df1[(df1["label"]==1) & (df1["preds"]==1)].index,1)
    img_path = df1.iloc[ind[0]]["path"]
    save_path = os.path.join(save_path_dir, img_path.split("/")[-3]+"-"+ img_path.split("/")[-1])
    save_gradcam(img_path,save_path, cmap='jet')
    
    
    
#img_path = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/Antibodies_detection/codes/tmi2022/antibodies_data/tiled_slide/PD152-DLB-EntCx-AnteriorHippo-C34-45_files/20.0/13_24.jpeg"

In [ ]:
df1.head(2)

In [ ]:

#model(input_tensor).argmax().cpu().numpy() == df.iloc[ind[0]]["label"]
img_resized,overlay = load_image_run_gradcam(img_path, model)
# 5. Plot
plt.figure(figsize=(6, 6))
plt.imshow(overlay)
plt.title("Grad-CAM Overlay")
plt.axis("off")
plt.show()

In [ ]:
img_path

In [ ]:
plt.figure(figsize=(6, 6))
plt.imshow(img_resized)
plt.axis("off")

In [ ]:
img_resized,overlayed = load_image_run_saliency_map(img_path, model)

# Saliency map
plt.figure(figsize=(6, 6))
plt.imshow(overlayed, cmap='hot')
plt.title('Saliency Map')
plt.axis('off')

# Ablation CAM

In [17]:
target_layer = model.encoder.features[-2]

In [18]:
def load_img_apply_ablationcam(img_path, model):
    img_resized, img_np, input_tensor = load_image(img_path)
    activations = []
    def forward_hook(module, input, output):
        activations.append(output.detach())
    hook_handle = target_layer.register_forward_hook(forward_hook)
    # Forward pass (with hook)
    output = model(input_tensor)
    pred_class = output.argmax().item()
    score_baseline = output[pred_class].item()

    # Get activations (shape: [1, C, H, W])
    activation_maps = activations[0]  # [1, C, H, W]
    B, C, H, W = activation_maps.shape
    # Compute class score drop after ablating each channel
    channel_scores = []

    with torch.no_grad():
        for i in range(C):
            ablated = activation_maps.clone()
            ablated[:, i, :, :] = 0  # Zero out i-th channel

            # Forward the rest of the model manually
            h = F.adaptive_avg_pool2d(ablated, 1).view(1, -1)
            x = model.classifier(h)

            class_score = x[0, pred_class].item()
            score_diff = score_baseline - class_score
            channel_scores.append(score_diff)
    # Build weighted sum of activation maps
    weights = torch.tensor(channel_scores).view(C, 1, 1).to(device)
    cam = (weights * activation_maps[0]).sum(dim=0)  # shape [H, W]
    cam = F.relu(cam)
    cam -= cam.min()
    cam /= cam.max()
    cam_np = cam.cpu().numpy()
    cam_resized = cv2.resize(cam_np, (img_np.shape[1], img_np.shape[0]))  # (W, H)
    # Convert CAM to color heatmap
    heatmap = plt.cm.jet(cam_resized)[..., :3]  # shape: (H, W, 3)

    # Blend with original image
    overlay = 0.7 * img_np + 0.3 * heatmap
    overlay = np.clip(overlay, 0, 1)
    hook_handle.remove()
    return img_resized,overlay 
    

In [25]:

img_resized,overlayed  = load_img_apply_ablationcam(img_path, model)

In [ ]:
plt.figure(figsize=(6, 6))
# Show result
plt.imshow(overlayed)
plt.title(f"AblationCAM - Class {pred_class}")
plt.axis('off')
plt.show()

 